# SIAM whole-head labeling

Run the original SIAM v0.3 nnU-Net model on an unstripped whole-head image.

**Resources:** downloads a sample T1, templates and pretrained weights on first use.
CPU is the default; allow substantial memory and several minutes or longer.
Set `t1_path` to an absolute path to use your own scalar 3-D T1 MRI.

## Additional installation and resources
Install `python -m pip install -e '.[notebooks,siam]'` before running.
The first inference downloads a **2.68 GB** archive and extracts the models
into the ANTsTorch cache. Allow disk space for both archive and checkpoints.
CPU inference can be very slow and needs substantial RAM.

Use the whole image with correct geometry. SIAM handles its own normalization
and resampling; no brain extraction, N4 correction or template registration
should be applied for this example. The sample T1 is used as distributed.

Install ANTsTorch and notebook tools from the repository root before opening this notebook:
```bash
python -m pip install -e '.[notebooks]'
python -m ipykernel install --user --name python3 --display-name "Python 3 (ANTsTorch)"
```
Select that kernel and run the cells in order. All examples default to CPU.

In [ ]:
import torch  # Import before ANTs to avoid OpenMP conflicts on macOS.
import numpy as np
import matplotlib.pyplot as plt
import ants
import antstorch

device = "cpu"
antstorch.set_default_device(device)

## Load the sample MRI or choose a local image

In [ ]:
from pathlib import Path

# Set an absolute path to use your own scalar 3-D T1 MRI instead.
t1_path = None
if t1_path is None:
    t1_path = antstorch.get_antstorch_data("mprage_hippmapp3r")
t1 = ants.image_read(str(t1_path)).clone("float")
assert t1.dimension == 3 and not t1.has_components
assert np.isfinite(t1.numpy()).all()
print("Input:", t1_path, "shape:", t1.shape, "spacing:", t1.spacing)

## Run published default inference
`number_of_folds=None` uses all published folds; `voxel_size=None` respects
the image's physical spacing. Requesting 18 probability images adds substantial
memory and computation, so only the segmentation is requested here.

In [ ]:
result = antstorch.siam_labeling(
    t1, return_probability_images=False, number_of_folds=None,
    voxel_size=None, device=device, verbose=True,
)
segmentation = result["segmentation_image"]

values = segmentation.numpy()
assert segmentation.shape == t1.shape
assert np.isfinite(values).all()
np.testing.assert_allclose(values, np.rint(values), atol=1e-6)
np.testing.assert_allclose(segmentation.spacing, t1.spacing)
np.testing.assert_allclose(segmentation.origin, t1.origin)
np.testing.assert_allclose(segmentation.direction, t1.direction)
assert (values > 0).any()

In [ ]:
label_values = np.array(list(result["labels"].values()), dtype=int)
assert np.isin(values, label_values).all()
ants.plot(t1, overlay=segmentation, axis=2, nslices=6,
          overlay_cmap="jet", overlay_alpha=0.5)
output_dir = Path("example_outputs/siam_labeling")
output_dir.mkdir(parents=True, exist_ok=True)
ants.image_write(segmentation, str(output_dir / "siam_labels.nii.gz"))
voxel_volume = float(np.prod(t1.spacing))
for name, label in result["labels"].items():
    volume = np.count_nonzero(values == label) * voxel_volume
    print(f"{label}: {name}: {volume:.1f} cubic physical units")
print("Outputs (matching files are overwritten):", output_dir.resolve())

Inspect the head, skull and brain structures separately. Using fewer folds or
`voxel_size=0.75` changes the published default inference; assess the effect
on your own data before using either speed option.

See the [SIAM guide](https://github.com/ANTsX/ANTsTorch/blob/main/docs/siam_labeling.md).
Cite the [SIAM paper](https://arxiv.org/abs/2605.02737) and
[model record](https://doi.org/10.5281/zenodo.17733431) when using the CC BY 4.0 weights.